# N067 · 网格路径回溯与访问恢复

**目标：** 区分同一路径访问和全局访问限制。

**先修：** N018, N065。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 四邻域；边界；visited；路径状态恢复；剪枝；分支因子。

**配套讲解来源：** [同名逐章意见](../../comment/067_grid_backtracking.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章我们把回溯搬到二维网格上，解决两道经典题：

**题一：单词搜索（Word Search）。** 给你一个字符网格和一个单词，问你能否在网格里"邻接着"走出这个单词。规则是：从一个格子出发，每次只能走到上下左右相邻的格子，同一个格子在同一条路径里不能走第二次。notebook 里的网格是：

```
A B C E
S F C S
A D E E
```

单词 `'ABCCED'` 能找到：A(0,0)→B(0,1)→C(0,2)→C(1,2)→E(2,2)→D(2,1)，每一步都相邻且没有格子重复。单词 `'SEE'` 也能找到：S(1,3)→E(2,3)→E(2,2)。但单词 `'ABCB'` 找不到：A(0,0)→B(0,1)→C(0,2 或 1,2) 之后需要一个还没用过的 B，而网格里唯一的 B 在 (0,1)，已经在路径上了——同一路径不能二次使用格子，这就是卡死的原因。

**题二：Unique Paths III。** 网格里标了起点 1、终点 2、空格 0、障碍 -1，问你有多少条路能"从起点出发、每个可走格子恰好经过一次、最后停在终点"。notebook 的例子：

```
1 0 0 0
0 0 0 0
0 0 2 -1
```

可走格子共 9 个（含起点终点），答案是 2 条。注意规则比普通走格子严格得多：你不能提前路过终点再绕回来——一旦踩到终点，必须恰好已经走完所有格子，这次踩到才算一条解。

两题共同的核心点：**"访问标记"只属于当前这条路径**。走回头路时要把标记擦掉，否则别的路径就没法用这个格子了。这是本章标题"访问恢复"的含义，也是和普通 BFS/DFS"走过就永久标记"最大的区别。

## 2. 基础知识：先读懂这些词

- **四邻域**：一个格子的上、下、左、右四个相邻位置。代码里写成方向增量对 `[(1,0),(-1,0),(0,1),(0,-1)]`，加上当前坐标 (r,c) 就得到邻居坐标。
- **边界检查**：`0<=r<rows and 0<=c<cols`，防止"走出棋盘"。网格题的第一大坑就是忘记邻居可能出界，比如 (0,0) 的上邻居 (-1,0) 不存在。
- **visited（访问标记）**：记录哪些格子已在当前路径上，避免同一路径踩同一格两次。本章两个实现都没有单独建 visited 数组，而是直接在棋盘上做记号——单词搜索把走过格子的字符临时改成 `None`，路径题把走过格子临时改成 `-1`（借用障碍的含义：本路径内不可再入）。棋盘本身就是最好的 visited。
- **路径状态恢复**：递归返回前必须把格子改回原值。如果不改，一条失败路径用过的格子会被永久封锁，后面所有路径都会因此漏解——练习 2 就是让你亲手构造这种反例。
- **剪枝**：提前砍掉死分支。单词搜索里的"当前格字符必须等于 word[i]"就是最锋利的一刀；边界检查也算一种。
- **分支因子**：每个格子的下一步最多 4 种选择。它决定搜索树有多"胖"：走 L 步最多 4ᴸ 条路径（实际因不能回头，第一步之后每步至多 3 个新方向）。
- **try/finally**：Python 语法，意思是"try 块不管正常结束还是抛异常，finally 块都执行"。这里用它保证"恢复格子"这个清理动作万无一失，即使递归中途抛异常，棋盘也不会留下脏标记。

## 3. 思路推导：从小例子开始

**Step 1：把"走网格"翻译成"每步选一个方向"。** 在 `visit(r,c,i)` 里，(r,c) 是当前格子，i 是已经匹配到的单词下标。四个方向逐一尝试，任何一个成功就整体成功。

**Step 2：用棋盘自身当 visited。** 进入格子前把它的字符存进 `saved`，把格子改成 `None`；这样邻居递归时 `board[r][c]!=word[i]` 自动把"已走过"的格子挡住；返回时用 finally 恢复。

**Step 3：单词搜索从每个格子都试一遍起点。** 答案是"存在一条路径"，所以外层用 `any(...)` 把所有起点或起来。

**Step 4：路径计数题换终止条件。** `unique_paths_iii` 里到达终点 2 时返回 `int(used==count)`：只有"已走格数等于可走格数"时这一趟才算 1 条解，否则算 0（提前到终点，白走）。

**Step 5：计数的递归结构。** 每个格子的路径数 = 四个邻居的路径数之和，用 `sum(...)` 而不是 `any(...)`，因为我们要数所有走法而不是找一个。

**手算演示 1：`exist(b,'SEE')`。**

1. 外层扫描找到 (1,3) 的 'S' 与 word[0] 匹配，进入 `visit(1,3,0)`：字符相等、i=0 不是末位，把 (1,3) 改成 None。
2. 试四个邻居找 'E'（word[1]）：(0,3) 是 'E'！进入 `visit(0,3,1)`：它也不是末位，改 None，四邻居找 word[2]='E'：(1,3) 是 None（被挡），(0,2) 是 'C'，出界——失败，恢复 (0,3)。
3. 回到 (1,3) 的另一个邻居 (2,3) 是 'E'：进入 `visit(2,3,1)`，改 None，找 word[2]='E'：邻居 (2,2) 是 'E'，进入 `visit(2,2,2)`：i==len(word)-1，直接返回 True！
4. True 一路向上传递，`any(...)` 收到 True；finally 逐层把三个格子恢复成 'S','E','E'。“运行示例”部分 的表格验证了这一点：搜索后首行 `''.join(b[0])` 仍是 `'ABCE'`。

**手算演示 2：`unique_paths_iii` 的 3×4 例子。** 先盘点：网格 3 行 4 列共 12 格，其中 (2,3) 是障碍 -1，其余 11 格（含起点 1 和终点 2）都算可走，所以 count=11。要求是：从 (0,0) 出发恰好走过 11 个格子、最后一步落在 (2,2)。两条解是：

- 解一（横向蛇形）：(0,0)→(0,1)→(0,2)→(0,3)→(1,3)→(1,2)→(1,1)→(1,0)→(2,0)→(2,1)→(2,2)。
- 解二（绕圈形）：(0,0)→(1,0)→(2,0)→(2,1)→(1,1)→(0,1)→(0,2)→(0,3)→(1,3)→(1,2)→(2,2)。

你可以逐格核对：两条路径都恰好 11 步覆盖 11 格、无重复、终点收尾。同时体会两个细节——若中途路过 (2,2) 再绕出去，那趟的 used 不足 11，按 `int(used==count)` 记 0 条；障碍 (2,3) 从一开始就不计入 count，也没人能踩进去。

## 4. 核心代码：exist

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def exist(board, word):
    if not word:
        return True
    if not board or not board[0]:
        return False
    rows, cols = (len(board), len(board[0]))

    def visit(r, c, i):
        if not (0 <= r < rows and 0 <= c < cols) or board[r][c] != word[i]:
            return False
        if i == len(word) - 1:
            return True
        saved = board[r][c]
        board[r][c] = None
        try:
            return any((visit(r + dr, c + dc, i + 1) for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]))
        finally:
            board[r][c] = saved
    return any((visit(r, c, 0) for r in range(rows) for c in range(cols)))
```

### 逐段读懂代码

### 4.1 `exist(board, word)`

（对应程序见下方分段实现与完整代码。）

- `if not word: return True`：空单词在任何棋盘（甚至空棋盘）上都算找到——零个字符无需匹配。先于空棋盘检查，顺序有讲究。
- `if not board or not board[0]: return False`：空棋盘（或每行为空）不可能走出非空单词。
- `if not (0<=r<rows and 0<=c<cols) or board[r][c]!=word[i]: return False`：一行干两件事——出界检查 + 字符匹配（含"格子被标记成 None"的情况，None 永远不等于字符，天然挡回头路）。
- `if i==len(word)-1: return True`：能走到这里说明 board[r][c]==word[i] 已成立，若 i 已是最后一个下标，整个单词匹配完成。
- `saved=board[r][c]; board[r][c]=None`：choose——踩上格子并做标记。
- `return any(visit(...) for dr,dc in [...])`：explore——四个方向逐个试；`any` 短路，第一个成功就返回 True，后面的方向不再试。
- `finally: board[r][c]=saved`：undo——无论 any 的结果如何都恢复原字符。放 finally 而不是普通下一行，是为了保证异常情况下棋盘也能复原。
- `return any(visit(r,c,0) for r in range(rows) for c in range(cols))`：外层把每个格子当起点试一遍，双重 for 展平成生成器交给 any。

### 4.2 `unique_paths_iii(grid)`

（对应程序见下方分段实现与完整代码。）

- 第一段双重循环做**盘点**：`count` 统计所有非 -1 的格子数（起点、终点、空格都算可走），顺便找到起点 1 的坐标。
- `if start is None: raise ValueError('a start cell is required')`：输入契约——没有起点这题无定义，直接报错而不是返回 0（0 会和"有起点但没有路径"混淆）。
- `if not (...) or grid[r][c]==-1: return 0`：出界或撞障碍（含"本路径已走过的格子"，因为走过会被临时改成 -1）都贡献 0 条路径。
- `if grid[r][c]==2: return int(used==count)`：踩到终点。`used` 是"包括起点在内已经踩过的格子数"；只有 used 恰好等于可走格总数时，这趟才是一条完整解，`int(True)`=1、`int(False)`=0。提前到终点 used<count，返回 0，相当于这条路线作废。
- `saved=grid[r][c]; grid[r][c]=-1`：choose——把当前格临时变成"障碍"，含义是"本路径已占用"。
- `return sum(visit(...) for ...)`：explore——四个方向的路径数**相加**（对比单词搜索的 any：找存在性用 any，数数量用 sum）。
- `finally: grid[r][c]=saved`：undo——恢复格子，供其他路径使用。
- `return visit(*start,1)`：从起点出发，初始已用格数为 1（起点自己算一个）；`*start` 把 (r,c) 拆成两个参数。

## 5. 分段实现：按顺序运行

**本章接口：** `exist(board, word)`、`unique_paths_iii(grid)`

### exist

In [1]:
def exist(board, word):
    if not word:
        return True
    if not board or not board[0]:
        return False
    rows, cols = (len(board), len(board[0]))

    def visit(r, c, i):
        if not (0 <= r < rows and 0 <= c < cols) or board[r][c] != word[i]:
            return False
        if i == len(word) - 1:
            return True
        saved = board[r][c]
        board[r][c] = None
        try:
            return any((visit(r + dr, c + dc, i + 1) for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]))
        finally:
            board[r][c] = saved
    return any((visit(r, c, 0) for r in range(rows) for c in range(cols)))

### unique_paths_iii

In [2]:
def unique_paths_iii(grid):
    if not grid or not grid[0]:
        return 0
    rows, cols = (len(grid), len(grid[0]))
    start = None
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != -1:
                count += 1
            if grid[r][c] == 1:
                start = (r, c)
    if start is None:
        raise ValueError('a start cell is required')

    def visit(r, c, used):
        if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] == -1:
            return 0
        if grid[r][c] == 2:
            return int(used == count)
        saved = grid[r][c]
        grid[r][c] = -1
        try:
            return sum((visit(r + dr, c + dc, used + 1) for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]))
        finally:
            grid[r][c] = saved
    return visit(*start, 1)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

b=[list('ABCE'),list('SFCS'),list('ADEE')]
show_table(['网格行','字符'],list(enumerate(b)))
show_table(['目标词','能否找到','查找后首行'],[(w,exist(b,w),''.join(b[0])) for w in ['ABCCED','SEE','ABCB']])

网格行,字符
0,"['A', 'B', 'C', 'E']"
1,"['S', 'F', 'C', 'S']"
2,"['A', 'D', 'E', 'E']"


目标词,能否找到,查找后首行
ABCCED,True,ABCE
SEE,True,ABCE
ABCB,False,ABCE


## 7. 正确性、适用条件与复杂度

每层路径标记排除已使用格子，回溯恢复让搜索树覆盖所有简单路径。终点仅在已用格数等于可走格数时计1，不把提前到达算解。

**代价：** 单词搜索粗界O(RC·4ᴸ)、O(L)栈；全格路径搜索最坏指数、O(RC)深度。try/finally用于恢复资源，不吞掉异常。

### 展开理解

**为什么每条可能的路径都被考虑到了？** 因为"走过即标 -1、返回即恢复"这套机制，让任何时刻递归看到的棋盘恰好反映"当前这条路径"的占用情况。一条路径走死了，恢复动作把它占用的格子全部释放，于是同一批格子可以被换一种顺序重新组合——所有"不重复经过任何格子的走法"（简单路径）都会被枚举一遍。

**为什么"提前到终点"不算解？** 判定条件 `used==count` 把"到达终点"和"用完全部格子"绑在一起：提前到终点时 used 不足 count，返回 0；这正好把"路过终点再绕行"的路线从答案里剔除，符合题意。

**复杂度（结合数字说）：** 单词搜索的粗上界是 O(RC·4ᴸ)：R×C 个起点候选，每个起点至多 4ᴸ 条长 L 的走法。比如 3×4 棋盘找 6 个字母，3×4×4⁶≈147k 次递归调用，瞬间完成；若 L=15、棋盘 6×6，粗界 36×4¹⁵≈10¹⁰ 就很吓人了——幸好第一步之后每步至多 3 个新方向且字符匹配剪掉绝大多数分支，实际远快于粗界。递归栈深最多 L。路径计数题本质是数哈密顿路径，最坏是指数级的（比如空旷的大棋盘），工作深度最多 O(RC)，12 格的例子深 11 层毫无压力。try/finally 在这里的作用是工程性的：保证任何情况下资源（格子状态）都被恢复，且它不会吞掉异常——异常照样往上抛，只是抛之前先做清理。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 逐条解释（配合 `deepcopy` 保存原棋盘）：

- `assert exist(b,'ABCCED') and exist(b,'SEE') and not exist(b,'ABCB')`：经典三连。前两个是"能找到"的正常值（'ABCCED' 考验走回头方向的 S 形路径，'SEE' 考验多个 E 的选择）；'ABCB' 是**核心反例**——唯一路径需要重复使用 (0,1) 的 B，考的就是"同一路径不能二次入格"。
- `assert b==old`：状态恢复测试。搜索结束后棋盘必须和搜索前一模一样，直接验证 undo/finally 是否尽责；如果实现忘了恢复，这个断言立刻红。
- `assert not exist([['A']],'AA')`：特殊值。单格棋盘找 'AA' 需要同一个格子用两次，必须返回 False——用最小输入打"路径内不可复用"这条规则。
- `assert unique_paths_iii(g)==2 and g==old`：正常值 + 恢复测试。3×4 例子的标准答案是 2；棋盘同样必须完好如初。
- 最后一大段 `for mask in range(16): ...`：**暴力对照测试**。在 2×3 网格上用 4 个比特枚举内部 4 个格子（`(0,1),(0,2),(1,0),(1,1)`）各自的"障碍/空格"状态，共 16 种布局；对每种布局，用带 `functools.cache` 的位掩码参考实现（`ref(v,seen)` 以"当前在哪个点、哪些点已访问"为状态记忆化数路径）算出标准答案，再和 `unique_paths_iii` 比对，并再次检查棋盘未被破坏。16 种布局穷举，等于把小规模的所有情况都对了一遍账。

In [4]:
from copy import deepcopy

b = [list('ABCE'), list('SFCS'), list('ADEE')]

old = deepcopy(b)

assert exist(b, 'ABCCED') and exist(b, 'SEE') and (not exist(b, 'ABCB'))

assert b == old

assert not exist([['A']], 'AA')

g = [[1, 0, 0, 0], [0, 0, 0, 0], [0, 0, 2, -1]]

old = deepcopy(g)

assert unique_paths_iii(g) == 2 and g == old

from functools import cache

for mask in range(16):
    g = [[1, 0, 0], [0, 0, 2]]
    inner = [(0, 1), (0, 2), (1, 0), (1, 1)]
    for i, (r, c) in enumerate(inner):
        if mask >> i & 1:
            g[r][c] = -1
    vertices = [(r, c) for r in range(2) for c in range(3) if g[r][c] != -1]
    index = {v: i for i, v in enumerate(vertices)}
    end = index[1, 2]

    @cache
    def ref(v, seen):
        if v == end:
            return int(seen == (1 << len(vertices)) - 1)
        r, c = vertices[v]
        total = 0
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nxt = (r + dr, c + dc)
            if nxt in index and (not seen >> index[nxt] & 1):
                total += ref(index[nxt], seen | 1 << index[nxt])
        return total
    old = deepcopy(g)
    assert unique_paths_iii(g) == ref(index[0, 0], 1 << index[0, 0]) and g == old

print('N067: 所有本章断言通过')

N067: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 禁止修改输入实现一次。

**练习 2：** 构造全局visited导致漏解的反例。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（禁止修改输入实现一次）**：思路是另建一个与棋盘同形的 `visited` 布尔矩阵（或 `set` 存坐标），把"标记/恢复"从改棋盘改成改 visited。提示三个要点：(1) 进入格子时 `visited[r][c]=True`，finally 里改回 False；(2) 边界与字符检查照旧，再加一条"visited 已标记则失败"；(3) 用 “自动测试”部分 的 `deepcopy` 对比法验证新版本既答案正确又不碰棋盘。手算示例可以直接用 'SEE'，画出三次标记与三次解除的时刻表。
- **练习 2（构造全局 visited 导致漏解的反例）**：思路——"全局 visited"指走过就永久标记、从不恢复。提示构造法：找一个格子，它必须被两条不同路径分别使用才能凑出两个答案。最直接的反例：1×3 网格 `[1,0,2]` 起终相邻只有 1 条路，不太好；换成 2×2 网格 `[[1,0],[0,2]]`：正确答案 2 条（先右再下 / 先下再右），它们共用中间两个格子。如果第一趟搜索后这两个格子被永久标脏，第二趟数出来 0，总答案 1≠2，漏解实锤。写完反例后用 `unique_paths_iii` 的正确输出 2 做对照即可。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def exist(board, word):
    if not word:
        return True
    if not board or not board[0]:
        return False
    rows, cols = (len(board), len(board[0]))

    def visit(r, c, i):
        if not (0 <= r < rows and 0 <= c < cols) or board[r][c] != word[i]:
            return False
        if i == len(word) - 1:
            return True
        saved = board[r][c]
        board[r][c] = None
        try:
            return any((visit(r + dr, c + dc, i + 1) for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]))
        finally:
            board[r][c] = saved
    return any((visit(r, c, 0) for r in range(rows) for c in range(cols)))

def unique_paths_iii(grid):
    if not grid or not grid[0]:
        return 0
    rows, cols = (len(grid), len(grid[0]))
    start = None
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != -1:
                count += 1
            if grid[r][c] == 1:
                start = (r, c)
    if start is None:
        raise ValueError('a start cell is required')

    def visit(r, c, used):
        if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] == -1:
            return 0
        if grid[r][c] == 2:
            return int(used == count)
        saved = grid[r][c]
        grid[r][c] = -1
        try:
            return sum((visit(r + dr, c + dc, used + 1) for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]))
        finally:
            grid[r][c] = saved
    return visit(*start, 1)

# 示例与回归测试
from copy import deepcopy

b = [list('ABCE'), list('SFCS'), list('ADEE')]

old = deepcopy(b)

assert exist(b, 'ABCCED') and exist(b, 'SEE') and (not exist(b, 'ABCB'))

assert b == old

assert not exist([['A']], 'AA')

g = [[1, 0, 0, 0], [0, 0, 0, 0], [0, 0, 2, -1]]

old = deepcopy(g)

assert unique_paths_iii(g) == 2 and g == old

from functools import cache

for mask in range(16):
    g = [[1, 0, 0], [0, 0, 2]]
    inner = [(0, 1), (0, 2), (1, 0), (1, 1)]
    for i, (r, c) in enumerate(inner):
        if mask >> i & 1:
            g[r][c] = -1
    vertices = [(r, c) for r in range(2) for c in range(3) if g[r][c] != -1]
    index = {v: i for i, v in enumerate(vertices)}
    end = index[1, 2]

    @cache
    def ref(v, seen):
        if v == end:
            return int(seen == (1 << len(vertices)) - 1)
        r, c = vertices[v]
        total = 0
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nxt = (r + dr, c + dc)
            if nxt in index and (not seen >> index[nxt] & 1):
                total += ref(index[nxt], seen | 1 << index[nxt])
        return total
    old = deepcopy(g)
    assert unique_paths_iii(g) == ref(index[0, 0], 1 << index[0, 0]) and g == old

print('N067: 所有本章断言通过')

N067: 所有本章断言通过


## 11. 代表题与迁移

- **79. Word Search**：对应 `exist`，练的是"棋盘即 visited + try/finally 恢复 + any 短路找存在性"，外加出界与字符匹配两刀剪枝。
- **980. Unique Paths III**：对应 `unique_paths_iii`，练的是"数路径用 sum、终点只在走满时计 1、障碍/走过统一用 -1 表达"。

两题合起来记住一句话：**找存在用 any，数数量用 sum；踩格先做记号，离开必须擦掉。**

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。